In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()
google_api_key = os.getenv('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature = 0.0)

In [20]:
from youtube_transcript_api import YouTubeTranscriptApi
from pytube import YouTube

def get_transcript(video_url):
    yt = YouTube(video_url)
    video_id = yt.video_id
    transcript = YouTubeTranscriptApi.get_transcript(video_id)
    full_text = " ".join([entry['text'] for entry in transcript])
    return full_text

In [21]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

video_url = "https://youtu.be/ylhcZZ7O3Tk?si=5lHtR0UsxaOMlD5A"
transcript_text = get_transcript(video_url)
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
docs = splitter.create_documents([transcript_text])

In [22]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import Chroma
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
db = Chroma.from_documents(docs, embeddings)

In [34]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

qa = ConversationalRetrievalChain.from_llm(
    llm = llm,
    retriever = db.as_retriever(),
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
)

C:\Users\HARDIK JAIN\AppData\Local\Temp\ipykernel_21532\764825284.py:7: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [40]:
import time
from langchain.prompts import PromptTemplate

print(qa.input_keys)
def chat():
    print("""
    What do you want to do:
    1. Ask a question
    2. Let me ask you 3 questions
    3. Exit the chat
    """)

    while True:
        a = input("Enter your choice: ")

        if a == "1":
            question = input("Enter your question: ")
            print("I'm thinking...")
            time.sleep(1)

            # ✅ Call QA system with question and history
            response = qa.invoke({
                "question": question,
            })
            print("Answer:", response["answer"])

        elif a == "2":
            print("I'll ask you 3 questions to test your understanding...")

            # ✅ Generate test questions
            qa_prompt = PromptTemplate.from_template(
                "You are a quiz bot. Based on this video transcript:\n\n{transcript}\n\n"
    "Generate **only 3 questions** to test the user's understanding. "
    "Do not provide any answers — only ask questions clearly."
            )
            prompt_text = qa_prompt.format_prompt(transcript=transcript_text).to_string()

            generated_questions = llm.invoke(prompt_text).content

            print("Answer the following questions:\n")
            print(generated_questions)

            user_response = input("\nYour answer: ")


            # ✅ Generate model's answer
            model_answer = llm.invoke(generated_questions).content

            # ✅ Grading prompt
            grade_prompt = PromptTemplate.from_template(
                "You are a grading assistant.\n"
                "Question: {question}\n"
                "User Answer: {answer}\n"
                "Model Answer: {model_answer}\n"
                "Now, give a score out of 10 and a short reason."
            )

            grade_input = grade_prompt.format_prompt(
                question=generated_questions,
                answer=user_response,
                model_answer=model_answer
            ).to_string()

            grade_response = llm.invoke(grade_input).content


            print("\nYour Score and Feedback:")
            print(grade_response)

        else:
            print("Goodbye! 👋")
            break

chat()


['question', 'chat_history']

    What do you want to do:
    1. Ask a question
    2. Let me ask you 3 questions
    3. Exit the chat
    
I'm thinking...
Answer: A variable is good for associating values with names, so you can reuse them in your program. The values of these variables might change.
I'll ask you 3 questions to test your understanding...
Answer the following questions:

1.  What is a variable, and why are variables useful in programming?
2.  Explain the difference in how the `+` operator functions when used with string variables versus number variables, providing an example for each.
3.  The video demonstrates reassigning values to variables. Describe a scenario where reassigning a variable's value would be beneficial in a program.

Your Score and Feedback:
Okay, here's a breakdown of the user's answer and a score:

**Strengths:**

*   **Correctly identifies the core concept of variables:** The user understands that variables are used to store and track information that